# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, overview, and exploration of the FAIR² dataset using the `mlcroissant` library, referencing dataset entities by their `@id` values as per best practices.

### Dataset Source
Dataset is described using a Croissant schema, available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and prepare access to record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset Croissant URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load metadata and dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review record sets, fields, and columns, referencing their `@id` identifiers. This helps understand the available data structure before extracting records.

In [ ]:
# List all record sets and their @id
record_sets = list(dataset.record_sets)
print("Available record sets in dataset (by @id):")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}")
    print(f"    name: {getattr(rs, 'name', '[no name]')}")
    for field in rs.fields:
        print(f"    Field @id: {field.id}   name: {getattr(field, 'name', '[no name]')}")
    for col in getattr(rs, 'columns', []):
        print(f"    Column @id: {col.id}")
    print("")

To preview the records in a given record set, pass its `@id` (string) to `dataset.records()`:

`for rec in dataset.records(record_set=record_set_id): ...`

In [ ]:
# Preview the first few records of each record set (using their @id)
for rs in record_sets:
    print(f"--- Records from RecordSet @id: {rs.id} ---")
    records_preview = []
    for i, rec in enumerate(dataset.records(record_set=rs.id)):
        records_preview.append(rec)
        if i == 2:
            break
    if records_preview:
        for r in records_preview:
            print(r)
    else:
        print("  [No records found]")
    print("")

## 3. Data Extraction
Load the records from each record set (referenced by its `@id`) into pandas DataFrames for further analysis.

In [ ]:
# Gather record set @id's
record_set_ids = [rs.id for rs in record_sets]

# DataFrames by record set @id
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if records:
        dataframes[rsid] = pd.DataFrame(records)

print("DataFrames available for these record_set @id's:")
for k in dataframes.keys():
    print(f"  {k}")

# Show an example DataFrame's columns and head
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns in DataFrame for RecordSet @id: {example_record_set_id}")
    print(dataframes[example_record_set_id].columns.tolist())
    dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Demonstrate processing: Filter, normalize, and group using column and field `@id`s. First, choose a numeric field present in the selected DataFrame.

> _Note_: All fields/columns are referenced by their `@id` to comply with Croissant best practices.

In [ ]:
# Select example record set for EDA
record_set_id = example_record_set_id  # from above; replace if you know the primary data table's @id
df = dataframes[record_set_id]

# Display available fields/columns for EDA
print(f"Columns available (by @id) in {record_set_id} DataFrame:")
print(df.columns.tolist())

# Select a likely numeric field by inspecting columns (if dataset is loaded)
# For this example, try with the first numeric-looking field
import numpy as np

numeric_field_id = None
for col in df.columns:
    if np.issubdtype(df[col].dropna().__class__, np.number):
        numeric_field_id = col
        break
    # Fallback: Try to convert a column to numeric and see if it succeeds
    try:
        pd.to_numeric(df[col])
        numeric_field_id = col
        break
    except Exception:
        continue

if numeric_field_id is None:
    print("No obvious numeric field found. Using the first column for demonstration.")
    numeric_field_id = df.columns[0]

# Attempt numeric conversion (in-place for EDA)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter numeric values above a threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
display(filtered_df.head())

# Normalize numeric field
norm_col = f"{numeric_field_id}_normalized"
if len(filtered_df) > 0:
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())
else:
    print("No records above the threshold.")

# Group by a categorical field (try to find one that isn't the numeric col)
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and df[col].dtype == object:
        group_field_id = col
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field using matplotlib/seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

if group_field_id and group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} grouped by {group_field_id}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, and analyze tabular data from a Croissant dataset using the `mlcroissant` Python library. All record sets and fields were referenced by their `@id` throughout. Explore the provided DataFrames for deeper statistical analysis and modeling tailored to your domain needs.